# 01 — Data Preparation

This notebook retrieves and prepares bioactivity data for the human Nav1.7 sodium channel (SCN9A) from ChEMBL.

The goal is to construct a clean and reproducible dataset for subsequent exploratory analysis, machine learning, and SHAP-based interpretation.

### Workflow
1. Retrieve Nav1.7 bioactivity records from ChEMBL
2. Filter and standardize IC50 measurements
3. Validate molecular structures
4. Assess data quality and repeated measurements
5. Calculate pIC50 values
6. Generate molecular descriptors
7. Export the cleaned dataset for downstream analysis

In [2]:
# Install ChEMBL web resource client
%pip install -q chembl_webresource_client

## 1. Retrieve Nav1.7 Bioactivity Data from ChEMBL

Nav1.7 (SCN9A) is a voltage-gated sodium channel involved in pain signaling and is an important target for analgesic drug discovery.

In this section, the ChEMBL database is queried to identify the human Nav1.7 single-protein target. All bioactivity records associated with this target are then retrieved for subsequent filtering and quality assessment.

## Run Configuration

During development, `TEST_MODE` can be enabled to retrieve only a small number of ChEMBL records. This allows the data-processing pipeline to be tested quickly without repeatedly downloading the complete dataset.

Set `TEST_MODE = False` when generating the final research dataset.

In [4]:
# =========================
# RUN CONFIGURATION
# =========================

TEST_MODE = True       # True = retrieve a small sample; False = retrieve all records
TEST_SIZE = 10

In [5]:
from chembl_webresource_client.new_client import new_client
import pandas as pd

# Connect to target API endpoint
target_api = new_client.target

# Search ChEMBL for Nav1.7 targets
target_query = target_api.search('Nav1.7')
targets_df = pd.DataFrame.from_dict(target_query)

# Filter for human single-protein targets
human_target = targets_df[
    (targets_df['target_type'] == 'SINGLE PROTEIN') &
    (targets_df['organism'] == 'Homo sapiens')
]

# Inspect matching targets before selecting the target ID
display(human_target[['target_chembl_id', 'pref_name', 'organism', 'target_type']])

,target_chembl_id,pref_name,organism,target_type
2,CHEMBL4296,Sodium channel protein type 9 subunit alpha,Homo sapiens,SINGLE PROTEIN


### Select the Human Nav1.7 Target

The filtered target table contains the human single-protein matches returned by the ChEMBL search.

After verifying that the first row (`index 0`) corresponds to the human Nav1.7 sodium channel, its ChEMBL target ID is selected for the subsequent bioactivity query.

Using `.iloc[0]` selects the first row by position, and `['target_chembl_id']` extracts its ChEMBL target identifier.

In [6]:
# Select Nav1.7 target
target_chembl_id = human_target.iloc[0]['target_chembl_id']

print(f"Target ID: {target_chembl_id}")

# Connect to activity API endpoint
activity_api = new_client.activity

# Query activities associated with Nav1.7
activity_query = activity_api.filter(
    target_chembl_id=target_chembl_id
)

# Retrieve either a small test sample or the complete dataset
if TEST_MODE:
    res = activity_query[:TEST_SIZE]
    print(f"TEST MODE: retrieving only {TEST_SIZE} records")
else:
    res = activity_query
    print("FULL MODE: retrieving all available records")

# Convert results to DataFrame
df = pd.DataFrame.from_dict(res)

print(f"Records retrieved: {len(df):,}")

Target ID: CHEMBL4296
TEST MODE: retrieving only 10 records
Records retrieved: 10


In [10]:
# Save raw dataset
df.to_csv('../data/raw/chembl_nav17_raw.csv', index=False)

OSError: Cannot save file into a non-existent directory: '../data/raw'